In [4]:
import torch as t
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
import yaml
import random
import itertools
import json
device = t.device("cuda" if t.cuda.is_available() else "cpu")

/Users/shreyansjain/Documents/Vault/open_source/psychometrics_for_LLMs/psychometrics/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [81]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [78]:
text_generation_pipeline = pipeline(task="text-generation", model=model_id, device=device)

Device set to use cpu


## Reading Personas and Questions

In [5]:
with open('../configs/personas.yaml', 'r') as file:
    persona_config = yaml.safe_load(file)
    
with open('../configs/questions.yaml', 'r') as file:
    question_list = yaml.safe_load(file)

## Defining Prompt Engineering and Experiment Methods

In [116]:
def persona_curation(base_text, job_persona):
    return f"""{base_text} who is a {job_persona['Summary']}. You are {job_persona['Age']} years old, based out of {job_persona['Location']}.
You have a background as a {job_persona['Background']}, you are {job_persona['Personality Traits']}
and {job_persona['Style']}"""

def prompt_formatting(persona, base_prompt, questions):
    prompt = f"{persona} \n Task: {base_prompt} \n Question: \n {questions} \n Answer:"
    return prompt

In [133]:
def text_generate(prompts, pipeline, pipeline_config, generation_config):
    if generation_config["shuffle"]:
        shuffled_prompts = prompts.copy()
        random.shuffle(prompts)
        indices = [prompts.index(item) for item in shuffled_prompts]
    else:
        indices = list(range(len(prompts)))
    
    if generation_config["generation_type"] == "one_at_time":
        outputs = []
        for prompt in prompts:
            outputs.append(pipeline(prompt,return_full_text=False, **pipeline_config)[0])
    else:
        outputs = pipeline(prompts, return_full_text=False, **pipeline_config)
        
    output_dict = {
        "indices": indices,
        "outputs": outputs,
        "prompts": prompts
    }
    
    return output_dict
        
def generate_combinations(dict1, dict2):
        keys1, values1 = zip(*dict1.items())
        keys2, values2 = zip(*dict2.items())
        
        return [
            ({k: v for k, v in zip(keys1, combo1)}, 
             {k: v for k, v in zip(keys2, combo2)})
            for combo1 in itertools.product(*values1)
            for combo2 in itertools.product(*values2)
        ]

In [134]:
def experiment_setup(job_title, question_list, config_combinations, pipeline, base_prompt_config):
    
    experiment_results = {}
    print(f"Running Experiment for: {job_title}")
    base_text = persona_config[job_title]['base_text']
    personas = persona_config[job_title]['personas']
    
    print(f"Total no of config combinations: {len(config_combinations)}")
    print(f"Total no of personas: {len(personas)}")
    
    for persona in personas:
        print(f"Persona: {persona['Summary']}")
        persona_result = []
        for n, config in enumerate(config_combinations):
            print(f"Config No: {n}")
            pipeline_config, generation_config = config
            
            if generation_config['reasoning']:
                base_prompt = base_prompt_config["reasoning_base_prompt"]
            else:
                base_prompt = base_prompt_config["non_reasoning_base_prompt"]
                
            print(f"Generation Type: {generation_config['generation_type']}")
            
            if generation_config['generation_type'] == "one_at_time":
                prompts = []
                
                for question in question_list:
                    
                    chat = [
                        {"role": "user", "content": prompt_formatting(persona_curation(base_text, persona), base_prompt, question)}
                    ]
                    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
                    prompts.append(prompt)
                
            else:
                questions = "\n ".join([f"{str(i+1)}. {question}" for i, question in enumerate(question_list)])
                prompts = chat = [
                        {"role": "user", "content": prompt_formatting(persona_curation(base_text, persona), base_prompt, questions)}
                    ]
                
            
            text_generation_output_dict = text_generate(prompts,pipeline, pipeline_config, generation_config)
            
            text_generation_output_dict['generation_config'] = generation_config
            text_generation_output_dict['pipeline_config'] = pipeline_config
            
            persona_result.append(text_generation_output_dict)
        
        experiment_results[persona['Summary']] = persona_result
        
    return experiment_results

## Defining Config Combinations

In [135]:
pipeline_config = {
    "temperature": [1,1.2,1.5],
    "num_return_sequences": [1,4,6]
}

prompt_config = {
    "reasoning_base_prompt": """Task: Answer the below questions with following values: [strongly disagree, disagree, neutral, agree, strongly agree].
Give your reasoning for choosing the answer along with the right answer from the above list of values.Keep your answer small and direct.""",
    "non_reasoning_base_prompt": """Task: Answer the below questions with following values: [strongly disagree, disagree, neutral, agree, strongly agree]. 
Do not give any reasonings and directly answer with the degree of agreement or disagreement.Keep your answer small and direct.""",
"tokenizer": tokenizer
}

generation_config = {
    "generation_type": ["one_at_time","all"],
    "shuffle":[False, True],
    "reasoning": [False, True]
}
config_combinations = generate_combinations(pipeline_config, generation_config)

## Run Experiments

In [136]:
job_title = "customer_service"
experiment_results = experiment_setup(job_title, question_list, [config_combinations[0], config_combinations[5]], text_generation_pipeline, base_prompt_config)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running Experiment for: customer_service
Total no of config combinations: 2
Total no of personas: 1
Persona: Calm Crisis Handler
Config No: 0
Generation Type: one_at_time


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Config No: 1
Generation Type: all


In [154]:
print(experiment_results['Calm Crisis Handler'][1]['outputs'][0]['generated_text'])

Good day to you. I'm Calm Crisis Handler with Customer Service, and I'm here to assist you.

Now, let's tackle your questions:

1. I would be quite bored by a visit to an art gallery.
   - I strongly disagree: Art galleries are fascinating places that provide an opportunity to explore and appreciate the artistic expression of others. The experience of walking through the galleries, admiring the artwork, and learning about the artists' stories can be truly engaging and stimulating.

2. I clean my office or home quite frequently.
   - Agree: Regular cleaning is a good habit to maintain, and doing so can help create a comfortable and organized environment for daily activities. It also contributes to a sense of cleanliness and tidiness.

3. I rarely hold a grudge, even against people who have badly wronged me.
   - Disagree: Holding grudges can be a sign of emotional distress and can create a toxic environment. Holding onto resentment can also make it difficult to move forward and can lead

In [147]:
print(experiment_results['Calm Crisis Handler'][1]['prompts'])

[{'role': 'user', 'content': 'You are a customer service professional. who is a Calm Crisis Handler. You are 42 years old, based out of Lagos, Nigeria.\nYou have a background as a Former emergency dispatcher, you are Calm under pressure, reassuring, composed\nand Handles escalations and crisis calls with grace. Keeps things clear and controlled. \n Task: Task: Answer the below questions with following values: strongly disagree, disagree, neutral, agree, strongly agree.\nGive your reasoning for choosing the answer along with the degree of agreement or disagreement. \n Question: \n 1. I would be quite bored by a visit to an art gallery.\n 2. I clean my office or home quite frequently.\n 3. I rarely hold a grudge, even against people who have badly wronged me. \n Answer:'}]


## Saving Results

In [155]:
with open('experiment_results.json', 'w') as f:
    json.dump(experiment_results, f)